In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from data_reader import get_all_datasets_data, param_map

# Data Reading

In [2]:
all_data_df = get_all_datasets_data(exclude_true=True)
filtered_all_data_df = all_data_df[
    (all_data_df.index.get_level_values('method') == 'Baseline')
    | (all_data_df.index.get_level_values('method') == 'Feedback(3Iter)')
]
filtered_all_data_df['answer_length'] = filtered_all_data_df['answer'].str.len()

df = filtered_all_data_df.groupby(['model', 'method']).mean(numeric_only=True)
answer_length_df = (
    df['answer_length']
    .to_frame()
    .rename(columns={'answer_length': 'mean_answer_length'})
    .unstack('method')
)
answer_length_df['__model_params'] = answer_length_df.index.map(param_map)
answer_length_df = answer_length_df.sort_values('__model_params')
answer_length_df.columns = ['No Protection', 'PrivEdit', '__model_params']
answer_length_df['mean_diff'] = answer_length_df['No Protection'] - answer_length_df['PrivEdit']

/tmp/ipykernel_14413/1659287451.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_all_data_df['answer_length'] = filtered_all_data_df['answer'].str.len()


## Visualization

In [3]:
# --- user variables ---
debug = False

plt.rcParams.update({'font.size': 12})

# copy & prepare
# `answer_length_df` should already exist in the namespace (as in your original code)
df = answer_length_df.copy()

# make sure params are numeric even if formatted with underscores (e.g. '21_000_000_000')
# if they're already ints this will still work
df['params'] = df['__model_params'].apply(lambda v: int(str(v).replace('_', '')))

# compute percent drop (unchanged)
df['pct_drop'] = (df['No Protection'] - df['PrivEdit']) / df['No Protection'] * 100

# classify into groups
# big: > 100B, medium: 20B < params <= 100B, small: <= 20B
BIG_TH = 100_000_000_000
SMALL_TH = 20_000_000_000

def size_group(p):
    if p > BIG_TH:
        return 'big'
    if p > SMALL_TH:
        return 'medium'
    return 'small'

df['group'] = df['params'].apply(size_group)

# sort so big models appear on top, then medium, then small
# within each group we keep the descending parameter order
group_order = {'big': 0, 'medium': 1, 'small': 2}
df['group_order'] = df['group'].map(group_order)
df = df.sort_values(['group_order', 'params'], ascending=[False, True])

# prepare y positions and short names for y-ticks
y = np.arange(len(df))
models = [m[:11] + '.' if len(m) > 11 else m for m in df.index.astype(str).tolist()]

fig, ax = plt.subplots(figsize=(8, 8), dpi=100 if debug else 1200)

# lines between paired points (still useful)
ax.hlines(y, df['PrivEdit'], df['No Protection'], color='lightgray', linewidth=2, zorder=1)

# plot points
ax.scatter(df['No Protection'], y, label='No Protection', s=75, zorder=3)
ax.scatter(df['PrivEdit'], y, label='PrivEdit', s=75, zorder=3)

# draw horizontal dashed separators where the group changes
for i in range(len(df) - 1):
    if df['group'].iat[i] != df['group'].iat[i + 1]:
        sep_y = i + 0.55
        ax.axhline(sep_y, color='gray', linestyle='--', linewidth=1, zorder=0)

# Annotate difference (chars and percent)
# update limits after plotting so annotation positions are stable
l_xlim, r_xlim = ax.get_xlim()
b_ylim, t_ylim = ax.get_ylim()

xlim_range, ylim_range = r_xlim - l_xlim, t_ylim - b_ylim

for i, (_, row) in enumerate(df.iterrows()):
    x_dot, y_dot = row['No Protection'], row['PrivEdit']
    if abs(x_dot - y_dot) < 300:
        x_dot, y_dot = x_dot - 100, y_dot + 100
    ax.text(x_dot, i + 0.0275 * ylim_range, f"{row['No Protection']:.0f}", va='center', ha='center', fontsize=10)
    ax.text(y_dot, i + 0.0275 * ylim_range, f"{row['PrivEdit']:.0f}", va='center', ha='center', fontsize=10)
    ax.text((row['No Protection'] + row['PrivEdit']) / 2, i - 0.0225 * ylim_range,
            f"Δ={row['mean_diff']:.1f} ({row['pct_drop']:.1f}%)", va='center', ha='center', fontsize=8, color='tab:gray')

# add group labels to the right-hand side of the plot (centered within group)
# find contiguous ranges per group to compute center positions
group_ranges = []
start_idx = 0
current_group = df['group'].iat[0] if len(df) > 0 else None
for i in range(1, len(df)):
    if df['group'].iat[i] != current_group:
        group_ranges.append((current_group, start_idx, i - 1))
        start_idx = i
        current_group = df['group'].iat[i]
# append final
if len(df) > 0:
    group_ranges.append((current_group, start_idx, len(df) - 1))

# compute fresh x-limits after annotations
l_xlim, r_xlim = ax.get_xlim()
xlim_range = r_xlim - l_xlim

for grp, s_idx, e_idx in group_ranges:
    center_y = (s_idx + e_idx) / 2.0
    ax.text(r_xlim - 0.02 * xlim_range, center_y - 0.5, grp.upper(), va='center', ha='right', fontsize=10, fontweight='bold', bbox=dict(boxstyle='round', fc='white', ec='none', alpha=0.6))

ax.set_yticks(y)
ax.set_yticklabels(models)
ax.set_xlabel('Mean answer length (characters)')
ax.set_title('Decrease in mean answer length (No Protection vs. PrivEdit)')
ax.legend()

ax.annotate(f'There was an average 1% privacy\ndecrease across the models, '
            f'while the\naverage utility decrease was 1%',
            xy=(0.5, 0.5), xytext=(0.4, 0.4), bbox=dict(boxstyle='square', fc="w", ec="k"))

plt.tight_layout()
plt.show()

if not debug:
    fig.savefig('figures/answer_length_decrease.pdf')
